In [ ]:
import os
import subprocess
import logging
import sys

def source_lmod_script(script_path):
    """
    Source an Lmod/module script and import environment variables into Python safely,
    suppressing terminal warnings.
    """
    # Use a non-interactive login shell (bash -l), redirect errors
    command = f'bash -l -c "source {script_path} >/dev/null 2>&1; printenv -0"'
    
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, shell=True)
    out, _ = proc.communicate()
    
    # Parse null-separated environment variables
    for env_var in out.split(b'\0'):
        if env_var:
            key, _, value = env_var.partition(b'=')
            os.environ[key.decode()] = value.decode()

# Example usage
M3_BUILD_DIR = "/home/henryi/scratch/venvs/.venv_sbi/bin/"
TUTORIAL_BUILD_DIR = M3_BUILD_DIR
source_lmod_script(f"{M3_BUILD_DIR}/setup.MaCh3.sh")
source_lmod_script(f"{TUTORIAL_BUILD_DIR}/setup.MaCh3Tutorial.sh")
os.environ["OMP_NUM_THREADS"] = "8"


my_stderr = sys.stderr = open('errors.txt', 'w')  # redirect stderr to file
get_ipython().log.handlers[0].stream = my_stderr  # log errors to new stderr
get_ipython().log.setLevel(logging.INFO)  # errors are logged at info level

In [ ]:
from mach3sbitools.inference.sbi_interface import MaCh3SBIInterface
from mach3sbitools.mach3_interface.mach3_simulator import MaCh3Simulator
from mach3sbitools.utils.logger import MaCh3Logger, get_logger
from mach3sbitools.utils.config import TrainingConfig, PosteriorConfig

from pathlib import Path
import uproot as ur
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import pyplot as plt
from tqdm import tqdm

logger = get_logger("mach3sbitools")
log_level='INFO'
log_file=None
MaCh3Logger("mach3sbitools", level=log_level, log_file=log_file)